In [1]:
#Install matplotlib and restart kernel
%pip install matplotlib
%pip uninstall bokeh -y
%pip install bokeh==2.4.2
%pip install "sagemaker>=2,<3"
%reset -f

# Install dependencies 
import boto3
import numpy as np
import pandas as pd
import sagemaker
import bokeh
import bokeh.io

from sagemaker.inputs import TrainingInput
from pprint import pprint
from sagemaker import image_uris
from sagemaker.session import Session
from sagemaker.tuner import IntegerParameter, CategoricalParameter, ContinuousParameter, HyperparameterTuner
from sagemaker.xgboost.estimator import XGBoost
from time import strftime
from bokeh.models import HoverTool
from bokeh.plotting import figure, show

sagemaker_session = sagemaker.Session()
role = sagemaker.get_execution_role()
region = boto3.Session().region_name
sess = boto3.Session()
sm = sess.client('sagemaker')

Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/18.5 MB ? eta -:--:--

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━ 13.9/18.5 MB 73.4 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 18.4/18.5 MB 70.7 MB/s eta 0:00:01

...

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━ 6/7 [sagemaker]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [sagemaker]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
autogluon-multimodal 1.5.0 requires nvidia-ml-py3<8.0,>=7.352.0, which is not installed.
autogluon-timeseries 1.5.0 requires chronos-forecasting<2.4,>=2.2.2, which is not installed.
autogluon-timeseries 1.5.0 requires einops<1,>=0.7, which is not installed.
autogluon-timeseries 1.5.0 requires peft<0.18,>=0.13.0, which is not installed.
skops 0.14.0 requires prettytable>=3.9, which is not installed.
autogluon-multimodal 1.5.0 requires fsspec[http]<=2025.3, but you have fsspec 2026.3.0 which is incompatible.
sagemaker-mlops 1.10.1 requires sagemaker-core>=2.10.1, but you have sagemaker-core 1.0.78 which is incompatible.
sagemaker-serve 1.10.1 requires sagemaker-core>=2.10.1, but you have sagemaker-core 1.0.78 which is incompatible.
sagemaker-studio-analytic

Note: you may need to restart the kernel to use updated packages.


sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [19]:
# Import the dataset 
s3 = boto3.resource('s3')
for buckets in s3.buckets.all():
    if 'labdatabucket' in buckets.name:
        bucket = buckets.name
print("Bucket: ", bucket)
prefix = 'scripts/data'
output_path = 's3://{}/{}/output'.format(bucket, prefix)

train_path = f"s3://{bucket}/{prefix}/adult_data_processed_train.csv"
validation_path = f"s3://{bucket}/{prefix}/adult_data_processed_validation.csv"

train_input = TrainingInput(train_path, content_type='text/csv')
validation_input = TrainingInput(validation_path, content_type='text/csv')

print(f'Training path: {train_path}')
print(f'Validation path: {validation_path}')

create_date = strftime("%m%d%H%M")
container = image_uris.retrieve(framework='xgboost',region=boto3.Session().region_name,version='1.2-1')
run_name = 'lab-3-run-{}'.format(create_date)
run_tags = [{'Key': 'lab-3', 'Value': 'lab-3-run'}]
job_name = 'lab-3-job-{}'.format(create_date)

Bucket:  labdatabucket-500
Training path: s3://labdatabucket-500/scripts/data/adult_data_processed_train.csv
Validation path: s3://labdatabucket-500/scripts/data/adult_data_processed_validation.csv


In [14]:
xgb_model = sagemaker.estimator.Estimator(
    container,
    role, 
    instance_count = 1, 
    instance_type ='ml.m5.xlarge',
    output_path = output_path,
    sagemaker_session = sagemaker_session,
    EnableSageMakerMetricsTimeSeries = True,
    tags = run_tags
)

In [15]:
hyperparameter_ranges = {
    'alpha': ContinuousParameter(0, 2),
    'eta': ContinuousParameter(0, 1),
    'max_depth': IntegerParameter(1, 10),
    'min_child_weight': ContinuousParameter(1, 10),
    'num_round': IntegerParameter(100, 1000)
}

In [16]:
objective_metric_name = 'validation:auc'
objective_type='Maximize'

In [17]:
tuner = HyperparameterTuner(
    estimator = xgb_model,
    objective_metric_name = objective_metric_name,
    hyperparameter_ranges = hyperparameter_ranges,
    objective_type = objective_type,
    max_jobs=12,
    max_parallel_jobs=2,
    early_stopping_type='Auto',
)

In [20]:
tuner.fit(
    {
        "train": train_input,
        "validation": validation_input
    },
    job_name=job_name,
    wait=True
)

No finished training job found associated with this estimator. Please make sure this estimator is only used for building workflow config


.

.

.

.

...

.

.

.

.

!

In [24]:
# Print the number of completed tuning jobs
tuning_job_result = sm.describe_hyper_parameter_tuning_job(
    HyperParameterTuningJobName=job_name
)

status = tuning_job_result["HyperParameterTuningJobStatus"]
if status != "Completed":
    print("Reminder: the tuning job has not been completed.")

job_count = tuning_job_result["TrainingJobStatusCounters"]["Completed"]
print("%d training jobs have completed" % job_count)

objective = tuning_job_result["HyperParameterTuningJobConfig"]["HyperParameterTuningJobObjective"]
is_minimize = objective["Type"] != "Maximize"
objective_name = objective["MetricName"]

6 training jobs have completed


In [23]:
# Get the best training job
if tuning_job_result.get("BestTrainingJob", None):
    print("Best model found so far:")
    pprint(tuning_job_result["BestTrainingJob"])
else:
    print("No training jobs have reported results yet.")

Best model found so far:
{'CreationTime': datetime.datetime(2026, 6, 3, 11, 31, 8, tzinfo=tzlocal()),
 'FinalHyperParameterTuningJobObjectiveMetric': {'MetricName': 'validation:auc',
                                                 'Value': 0.9141499996185303},
 'ObjectiveStatus': 'Succeeded',
 'TrainingEndTime': datetime.datetime(2026, 6, 3, 11, 33, 29, tzinfo=tzlocal()),
 'TrainingJobArn': 'arn:aws:sagemaker:us-east-1:146962103229:training-job/lab-3-job-06031123-008-90ab7d05',
 'TrainingJobName': 'lab-3-job-06031123-008-90ab7d05',
 'TrainingJobStatus': 'Stopped',
 'TrainingStartTime': datetime.datetime(2026, 6, 3, 11, 31, 50, tzinfo=tzlocal()),
 'TunedHyperParameters': {'alpha': '1.4456433637165826',
                          'eta': '1.0',
                          'max_depth': '1',
                          'min_child_weight': '5.845382724735384',
                          'num_round': '1000'}}


In [25]:
# Print the tuning metrics
tuner = sagemaker.HyperparameterTuningJobAnalytics(job_name)

full_df = tuner.dataframe()

if len(full_df) > 0:
    df = full_df[full_df["FinalObjectiveValue"] > -float("inf")]
    if len(df) > 0:
        df = df.sort_values("FinalObjectiveValue", ascending=is_minimize)
        print("Number of training jobs with valid objective: %d" % len(df))
        print({"lowest": min(df["FinalObjectiveValue"]), "highest": max(df["FinalObjectiveValue"])})
        pd.set_option("display.max_colwidth", None)  # Don't truncate TrainingJobName
    else:
        print("No training jobs have reported valid results yet.")

df

Number of training jobs with valid objective: 12
{'lowest': 0.5, 'highest': 0.9141499996185303}


,alpha,eta,max_depth,min_child_weight,num_round,TrainingJobName,TrainingJobStatus,FinalObjectiveValue,TrainingStartTime,TrainingEndTime,TrainingElapsedTimeSeconds
4,1.445643,1.000000,1.0,5.845383,1000.0,lab-3-job-06031123-008-90ab7d05,Stopped,0.91415,2026-06-03 11:31:50+00:00,2026-06-03 11:33:29+00:00,99.0
7,1.242939,0.528363,3.0,1.360401,981.0,lab-3-job-06031123-005-8a136837,Completed,0.91394,2026-06-03 11:28:08+00:00,2026-06-03 11:29:02+00:00,54.0
0,0.000000,0.567799,1.0,2.847297,1000.0,lab-3-job-06031123-012-4781e084,Stopped,0.91362,2026-06-03 11:33:48+00:00,2026-06-03 11:34:34+00:00,46.0
2,1.969564,0.744928,1.0,9.385531,1000.0,lab-3-job-06031123-010-82d7def6,Stopped,0.91309,2026-06-03 11:32:11+00:00,2026-06-03 11:33:01+00:00,50.0
10,0.580925,0.809489,3.0,6.752179,786.0,lab-3-job-06031123-002-c2e35779,Completed,0.91218,2026-06-03 11:24:57+00:00,2026-06-03 11:26:52+00:00,115.0
1,1.578362,0.830387,1.0,8.552124,385.0,lab-3-job-06031123-011-da2baec8,Stopped,0.91059,2026-06-03 11:33:24+00:00,2026-06-03 11:34:02+00:00,38.0
6,1.819209,0.573509,4.0,5.196198,1000.0,lab-3-job-06031123-006-734a9499,Stopped,0.90738,2026-06-03 11:30:25+00:00,2026-06-03 11:30:59+00:00,34.0
11,1.847654,0.802239,4.0,4.863361,960.0,lab-3-job-06031123-001-b9bb3c08,Completed,0.90443,2026-06-03 11:24:59+00:00,2026-06-03 11:27:04+00:00,125.0
3,0.554370,0.926212,1.0,6.026119,100.0,lab-3-job-06031123-009-56ef6246,Completed,0.90406,2026-06-03 11:31:15+00:00,2026-06-03 11:31:49+00:00,34.0
8,1.658232,0.598830,7.0,7.904282,888.0,lab-3-job-06031123-004-5933cd57,Completed,0.88796,2026-06-03 11:27:31+00:00,2026-06-03 11:28:25+00:00,54.0


In [26]:
# Plot the objective metric results against time
bokeh.io.output_notebook()

df = df.sort_values(by=['TrainingStartTime'], ascending=True)

# x = df['TrainingStartTime'].to_numpy()
x = df['TrainingStartTime']
y = df['FinalObjectiveValue'].to_numpy()

p = figure(
    title="Final Objective Value over Time", 
    width=900, height=400, 
    x_axis_label="TrainingStartTime",
    y_axis_label="FinalObjectiveValue",
    x_axis_type="datetime"
)

# add hover tool 
hover = HoverTool(tooltips=[
    ('FinalObjectiveValue', '@y'),
    ('TrainingStartTime', "@x{%T}")
    ], formatters={'@x': 'datetime'}) 
p.add_tools(hover) 


# p.circle(source=df, x="TrainingStartTime", y="FinalObjectiveValue")
p.line(x,y,color='green',line_width=2)
p.circle(x, y, fill_color ="red", line_color ="green", size=8) 

show(p)

Loading BokehJS ...

In [28]:
# Plot each of the hyperparameter ranges with the objective metric results
ranges = tuner.tuning_ranges
figures = []
for hp_name, hp_range in ranges.items():
    categorical_args = {}

    x = df[hp_name].to_numpy()
    y = df['FinalObjectiveValue'].to_numpy()

    # determine line of best fit
    par = np.polyfit(x, y, 1, full=True)
    slope=par[0][0]
    intercept=par[0][1]
    y_predicted = [slope*i + intercept  for i in x]        


    p = figure(
        width=500,
        height=500,
        title="Objective vs %s" % hp_name,
        x_axis_label=hp_name,
        y_axis_label=objective_name,
        **categorical_args,
    )

    # add hover tool 
    hover = HoverTool(tooltips=[
        ('FinalObjectiveValue', '@y'),
        (hp_name, '@x')
    ]) 
    p.add_tools(hover) 

    p.circle(source=df, x=hp_name, y="FinalObjectiveValue")
    p.line(x,y_predicted,color='green', line_width=2)
    figures.append(p)


show(bokeh.layouts.Column(*figures))